In [59]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
import pandas as pd

In [60]:
df = pd.DataFrame({
    "soil_moisture": [0.10, 0.15, 0.20, 0.25, 0.40, 0.60, 0.35, 0.18,
                      0.45, 0.05, 0.80, 0.27, 0.55, 0.70, 0.12, 0.30],
    "temperature_c": [34, 30, 26, 22, 28, 30, 19, 22,
                      35, 24, 33, 33, 21, 25, 20, 29],
    "sunlight_hours": [9, 8, 7, 4, 8, 10, 3, 10,
                       12, 5, 9, 11, 2, 6, 1, 9],
    "needs_water": [1, 1, 1, 0, 0, 0, 1, 1,
                    0, 1, 0, 1, 0, 0, 1, 1]
})

In [61]:
df

,soil_moisture,temperature_c,sunlight_hours,needs_water
0,0.10,34,9,1
1,0.15,30,8,1
2,0.20,26,7,1
3,0.25,22,4,0
4,0.40,28,8,0
5,0.60,30,10,0
6,0.35,19,3,1
7,0.18,22,10,1
8,0.45,35,12,0
9,0.05,24,5,1


In [62]:
X = df[['soil_moisture','temperature_c','sunlight_hours']]
y = df['needs_water']

**Normalizationg using minmax scaler**

In [63]:
X_min = X.min()
X_max = X.max()
X_scaled = (X - X_min) / (X_max - X_min + 1e-8)

In [64]:
# First split: Training + Temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: Validation + Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

**Creating a Model**

In [65]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(8,activation='relu'),
    layers.Dense(1,activation='sigmoid')
])

**Compile the model**

In [66]:
model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])

**Training the model**

In [67]:
history = model.fit(
    X_train.values,
    y_train.values,
    validation_data = (X_test.values, y_test.values),
    epochs = 100,
    batch_size = 4,
    verbose = 1
)

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 167ms/step - accuracy: 0.4545 - loss: 0.7132 - val_accuracy: 0.3333 - val_loss: 0.9314
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.4545 - loss: 0.7099 - val_accuracy: 0.3333 - val_loss: 0.9221
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.4545 - loss: 0.7065 - val_accuracy: 0.3333 - val_loss: 0.9140
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.4545 - loss: 0.7031 - val_accuracy: 0.3333 - val_loss: 0.9059
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.4545 - loss: 0.7013 - val_accuracy: 0.3333 - val_loss: 0.9003
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.4545 - loss: 0.6983 - val_accuracy: 0.3333 - val_loss: 0.8940
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.5455 - loss: 0.6959 - val_accuracy: 0.3333 - val_loss: 0.8870
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.5455 - loss: 0.6928 - val_accuracy: 0.3333 - val_loss

In [69]:
# Final Evaluation on Unseen Test Data
test_loss, test_accuracy = model.evaluate(
    X_test.values,
    y_test.values,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# Predictions
y_pred_probability = model.predict(X_test.values)

y_pred = (y_pred_probability >= 0.5).astype(int)

print("Predictions:")
print(y_pred.flatten())

print("Actual:")
print(y_test.values)

Test Loss: 0.6335324645042419
Test Accuracy: 0.6666666865348816
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Predictions:
[0 1 0]
Actual:
[1 1 0]
